# Hướng 2: CNN tự xây dựng

Phân loại biển báo giao thông Việt Nam thành 5 nhóm (P, W, R, I, S), ảnh đầu vào **32x32x3**.
Mạng CNN tự xây từ các khối `MyConvBlock` giống bài **04a** (Conv → BatchNorm → ReLU → Dropout → MaxPool), train từ đầu (không dùng trọng số có sẵn). Lớp Conv đầu tiên dùng kernel **7x7**.

Trước khi chạy, upload lên Google Drive thư mục `MyDrive/bienbao/`:
- `cnn_dataset_32.zip` (nén từ `Data_Processed/cnn_dataset_32`, tạo bằng `Data_Pipeline/build_cnn_crops.py`)

Chọn **Runtime → Change runtime type → T4 GPU**, rồi chạy lần lượt từng cell.
Model và lịch sử train được lưu vào `MyDrive/bienbao/ket_qua/`.

In [1]:
!nvidia-smi

Sun Nov  1 17:43:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.107.02             Driver Version: 550.107.02     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650 ...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   79C    P0             15W /   35W |       5MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os

try:  # Chạy trên Colab: lấy dataset từ Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/bienbao'
    if not os.path.exists('/content/cnn_dataset_32'):
        !unzip -q {DRIVE_DIR}/cnn_dataset_32.zip -d /content/
    DATA_DIR = '/content/cnn_dataset_32/'
    OUT_DIR = DRIVE_DIR + '/ket_qua/'
except ImportError:  # Chạy trên máy
    DATA_DIR = '../../Data_Processed/cnn_dataset_32/'
    OUT_DIR = '../Inference/'
os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
import glob
import json
import time

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as transforms
import torchvision.io as tv_io
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()

True

## Dữ liệu
Ảnh crop biển báo 32x32, chia sẵn train/val/test. Toàn bộ ảnh được đưa lên GPU một lần (giống bài 03).

In [4]:
IMG_HEIGHT = 32
IMG_WIDTH = 32
IMG_CHS = 3

# Tên thư mục = nhãn (lớp IE đã bỏ vì quá ít ảnh)
DATA_LABELS = ['0_P', '1_W', '2_R', '3_I', '5_S']
N_CLASSES = len(DATA_LABELS)

class MyDataset(Dataset):
    def __init__(self, data_dir):
        imgs = []
        labels = []
        for l_idx, label in enumerate(DATA_LABELS):
            data_paths = sorted(glob.glob(data_dir + label + '/*.png'))
            for path in data_paths:
                imgs.append(tv_io.read_image(path, tv_io.ImageReadMode.RGB))  # 3 x 32 x 32
                labels.append(l_idx)
        self.xs = (torch.stack(imgs).float() / 255).to(device)  # Normalize values from 0 to 1
        self.ys = torch.tensor(labels).to(device)

    def __getitem__(self, idx):
        return self.xs[idx], self.ys[idx]

    def __len__(self):
        return len(self.xs)

In [5]:
n = 64

train_data = MyDataset(DATA_DIR + 'train/')
train_loader = DataLoader(train_data, batch_size=n, shuffle=True)
train_N = len(train_loader.dataset)

valid_data = MyDataset(DATA_DIR + 'val/')
valid_loader = DataLoader(valid_data, batch_size=n)
valid_N = len(valid_loader.dataset)

test_data = MyDataset(DATA_DIR + 'test/')
test_loader = DataLoader(test_data, batch_size=n)
test_N = len(test_loader.dataset)

print('Train/Valid/Test:', train_N, valid_N, test_N)
train_data.xs.shape

Train/Valid/Test: 29317 3974 2259


torch.Size([29317, 3, 32, 32])

## Xây dựng model
Mỗi khối Conv giảm kích thước ảnh một nửa nhờ MaxPool: 32 → 16 → 8 → 4.

`MyConvBlock` viết dạng hàm trả về `nn.Sequential` (thay vì class) để file `.pth` lưu cả model có thể load lại ở notebook khác mà không cần định nghĩa lại.

In [6]:
def MyConvBlock(in_ch, out_ch, dropout_p, kernel_size=3):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size, stride=1, padding=kernel_size // 2),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(),
        nn.Dropout(dropout_p),
        nn.MaxPool2d(2, stride=2)
    )

In [7]:
flattened_img_size = 128 * 4 * 4

# Input 3 x 32 x 32
my_model = nn.Sequential(
    MyConvBlock(IMG_CHS, 32, 0, kernel_size=7),  # 32 x 16 x 16 (Conv đầu tiên 7x7)
    MyConvBlock(32, 64, 0.2),                    # 64 x 8 x 8
    MyConvBlock(64, 128, 0.2),                   # 128 x 4 x 4
    # Flatten to Dense Layers
    nn.Flatten(),
    nn.Linear(flattened_img_size, 512),
    nn.Dropout(.3),
    nn.ReLU(),
    nn.Linear(512, N_CLASSES)
)
my_model = my_model.to(device)

print(f'Tổng tham số: {sum(p.numel() for p in my_model.parameters()):,}')
my_model

Tổng tham số: 1,149,189


Sequential(
  (0): Sequential(
    (0): Conv2d(3, 32, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0, inplace=False)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (1): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (2): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (3): Flat

In [8]:
loss_function = nn.CrossEntropyLoss()
optimizer = Adam(my_model.parameters())

## Data augmentation
Giống bài 04a nhưng **không lật ngang** (`RandomHorizontalFlip`), vì lật biển "cấm rẽ trái" sẽ thành "cấm rẽ phải".

In [9]:
random_trans = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop((IMG_WIDTH, IMG_HEIGHT), scale=(.85, 1), ratio=(1, 1)),
    transforms.ColorJitter(brightness=.2, contrast=.2, saturation=.2),
])

## Hàm train / validate (giống bài 03)

In [10]:
def get_batch_accuracy(output, y, N):
    pred = output.argmax(dim=1, keepdim=True)
    correct = pred.eq(y.view_as(pred)).sum().item()
    return correct / N


def train(model):
    loss = 0
    accuracy = 0

    model.train()
    for x, y in train_loader:
        output = model(random_trans(x))
        optimizer.zero_grad()
        batch_loss = loss_function(output, y)
        batch_loss.backward()
        optimizer.step()

        loss += batch_loss.item()
        accuracy += get_batch_accuracy(output, y, train_N)
    loss = loss / len(train_loader)
    print('Train - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))
    return loss, accuracy


def validate(model, loader, N, name='Valid'):
    loss = 0
    accuracy = 0

    model.eval()
    with torch.no_grad():
        for x, y in loader:
            output = model(x)

            loss += loss_function(output, y).item()
            accuracy += get_batch_accuracy(output, y, N)
    loss = loss / len(loader)
    print('{} - Loss: {:.4f} Accuracy: {:.4f}'.format(name, loss, accuracy))
    return loss, accuracy


history = {'train_loss': [], 'train_acc': [], 'valid_loss': [], 'valid_acc': []}

def run_epochs(model, epochs):
    for epoch in range(epochs):
        print('Epoch: {}'.format(len(history['train_loss'])))
        train_loss, train_acc = train(model)
        valid_loss, valid_acc = validate(model, valid_loader, valid_N)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['valid_loss'].append(valid_loss)
        history['valid_acc'].append(valid_acc)

## Train
Tổng 40 epoch, bằng với Hướng 1 (5 + 35) để so sánh công bằng.

In [11]:
start = time.time()
run_epochs(my_model, 40)
train_time = time.time() - start
print(f'Thời gian train: {train_time / 60:.1f} phút')

Epoch: 0


Train - Loss: 0.3544 Accuracy: 0.8838
Valid - Loss: 0.1761 Accuracy: 0.9595
Epoch: 1


Train - Loss: 0.2131 Accuracy: 0.9307
Valid - Loss: 0.1306 Accuracy: 0.9595
Epoch: 2


Train - Loss: 0.1728 Accuracy: 0.9431
Valid - Loss: 0.0993 Accuracy: 0.9703
Epoch: 3


Train - Loss: 0.1459 Accuracy: 0.9525
Valid - Loss: 0.0757 Accuracy: 0.9771
Epoch: 4


Train - Loss: 0.1309 Accuracy: 0.9566
Valid - Loss: 0.0616 Accuracy: 0.9811
Epoch: 5


... (31 outputs left out)

Train - Loss: 0.0241 Accuracy: 0.9916
Valid - Loss: 0.0418 Accuracy: 0.9884
Epoch: 37


Train - Loss: 0.0255 Accuracy: 0.9917
Valid - Loss: 0.0378 Accuracy: 0.9884
Epoch: 38


Train - Loss: 0.0215 Accuracy: 0.9928
Valid - Loss: 0.0446 Accuracy: 0.9864
Epoch: 39


Train - Loss: 0.0248 Accuracy: 0.9922
Valid - Loss: 0.0491 Accuracy: 0.9864
Thời gian train: 2.9 phút


In [12]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history['train_acc'], label='Train')
plt.plot(history['valid_acc'], label='Valid')
plt.title('Accuracy'); plt.xlabel('Epoch'); plt.legend(); plt.grid(True)
plt.subplot(1, 2, 2)
plt.plot(history['train_loss'], label='Train')
plt.plot(history['valid_loss'], label='Valid')
plt.title('Loss'); plt.xlabel('Epoch'); plt.legend(); plt.grid(True)
plt.show()

/tmp/ipykernel_11395/1506016483.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Đánh giá trên tập test và lưu model

In [13]:
test_loss, test_acc = validate(my_model, test_loader, test_N, name='Test')

history['train_time_s'] = train_time
history['test_acc'] = test_acc
torch.save(my_model, OUT_DIR + 'custom_cnn.pth')
json.dump(history, open(OUT_DIR + 'custom_cnn_history.json', 'w'))
print('Đã lưu', OUT_DIR + 'custom_cnn.pth')

Test - Loss: 0.0170 Accuracy: 0.9947
Đã lưu ../Inference/custom_cnn.pth
